# Memorization test of the prior

How well can the prior memorize a small set of poses? For every size n in `SET_SIZES` (e.g. 1, 10, 100, 1000 poses), a fresh model is trained only on those n poses, and the log-likelihood is tracked on:
- **the n memorized poses** (the training data), and
- **a fixed held-out set** of `N_HELD_OUT` other poses that the model never sees (the reference).

What to expect:
- **Small n** (with n = 1 this is the classic "overfit a single batch" check): the log-likelihood of the memorized poses keeps rising far above the usual values (~135 on the test data, ~166 on the training data of the full training). The model concentrates its density on these poses, and the held-out log-likelihood falls.
- **Growing n**: the memorized log-likelihood gets lower and closer to the held-out one. This shows the capacity of the model.
- If even a single pose cannot be fitted well, the optimization or the model is broken.

The memorized sets are nested (the 10 poses contain the single pose, and so on), so only the size changes between the runs. The model uses the setup of the pretrained prior `id_0900`, with the same initialization (seed) for every n. The real pipeline of `SO3PriorTrainer` is used (model, optimizer, LR scheduler, gradient clipping, `train()`); only its data are replaced, via `AMASS(data=...)`. The batch is the whole memorized set, at most `BATCH_SIZE` poses.

The poses are drawn at random from the training split of the compressed AMASS (every 10th frame).

In [ ]:
import importlib
import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from hp.core.prior_training import make_config
from hp.data.amass import AMASS
from hp.data.amass_cfg import ORIGINAL_AMASS_SPLITS
from hp.utils.paths import REPO_ROOT

# ---- parameters ----------------------------------------------------------------------
BASE_CONFIG = REPO_ROOT / "experiments" / "pretrained_models" / "id_0900" / "config.yaml"  # model and training settings
ROTATIONS = "~/data/AMASS_rotations_c10"        # rotation files of the compressed AMASS; poses from its train split
OUT_DIR = "~/data/experiments/memorization_test"  # results (json, plot); no model checkpoints are saved
SET_SIZES = [1, 10, 100, 1000]  # numbers of memorized poses, one run each (10_000 adds about 1 h)
N_HELD_OUT = 2000               # never-seen poses, the reference
BATCH_SIZE = 2000               # upper limit; the batch is the whole set when it is smaller
MAX_ITERATIONS = 1000           # per run; the time per iteration grows with the batch (1000 poses: ~1.7 s on CPU)
VALID_INTERVAL = 50             # iterations between two measurements
DEVICE = "cpu"
NUM_THREADS = None              # e.g. 4 to leave CPU cores to another running notebook; None = torch default
SEED = 10                       # model initialization and training, the same for every n
DATA_SEED = 1                   # choice of the poses
QUICK = False                   # True: tiny run (sets of 1 and 10, 30 iterations) to check that the notebook works
# ----------------------------------------------------------------------------------------

if QUICK:
    SET_SIZES, N_HELD_OUT, MAX_ITERATIONS, VALID_INTERVAL = [1, 10], 200, 30, 10
if NUM_THREADS:
    torch.set_num_threads(NUM_THREADS)

ROTATIONS, OUT_DIR = Path(ROTATIONS).expanduser(), Path(OUT_DIR).expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HUPROSO_AMASS_ROTATIONS"] = str(ROTATIONS)  # this kernel only
os.environ["HUPROSO_AMASS_SPLIT"] = "original"
os.environ["HUPROSO_DEVICE"] = DEVICE
os.environ.pop("HUPROSO_SUBSET", None)
print(f"set sizes {SET_SIZES}, {N_HELD_OUT} held-out poses; {MAX_ITERATIONS} iterations, measured every {VALID_INTERVAL}; "
      f"{torch.get_num_threads()} CPU threads")

## The poses




One random permutation of the training split: the memorized set of size n is its first n poses (so the sets are nested), the held-out poses are taken from its end.

In [ ]:
names = ORIGINAL_AMASS_SPLITS["train"]
parts = [torch.load(ROTATIONS / f"{n}.pt")[:, :22].float() for n in names]  # quaternions [x, y, z, w], as AMASS.load_data
pool = torch.cat(parts)
del parts

order = np.random.default_rng(DATA_SEED).permutation(len(pool))
memorize_pool = AMASS(data=pool[order[: max(SET_SIZES)]])  # rotation matrices; the set of size n = its first n rows
held_out = AMASS(data=pool[order[-N_HELD_OUT:]])
print(f"training split: {len(pool):,} poses from {len(names)} datasets; up to {max(SET_SIZES):,} memorized, {N_HELD_OUT:,} held out")

## Training with the pipeline

`run(n)` builds an `SO3PriorTrainer` from the config of `id_0900` (same seed: the same initial model for every n), replaces its training and validation data by the n memorized poses, and calls its `train()`. At every validation it records the log-likelihood (per pose, higher is better) of the memorized poses and of the held-out poses.

In [ ]:
def mean_log_likelihood(trainer, dataset):
    """Mean log-likelihood of the poses of a dataset and the number of NaN values, without gradients."""
    trainer.pdf_estimator.eval()
    out = []
    with torch.no_grad():
        for start in range(0, len(dataset), BATCH_SIZE):
            out.append(trainer.forward_estimator(dataset.data[start : start + BATCH_SIZE].to(DEVICE)).cpu())
    trainer.pdf_estimator.train()
    ll = torch.cat(out).double().numpy()
    return float(np.nanmean(ll)), int(np.isnan(ll).sum())


def run(n):
    batch = min(n, BATCH_SIZE)
    cfg = make_config(BASE_CONFIG, exp_id=0, save_root=str(OUT_DIR), max_iterations=MAX_ITERATIONS,
                      valid_interval=VALID_INTERVAL, valid_wasserstein=False, device=DEVICE, seed=SEED,
                      save_ckpt_every=10**9,  # no checkpoints
                      overrides={"batch_size": batch, "num_epochs": 10**9,  # epochs must not cap the iterations
                                 "only_eval": True, "evaluate": False,
                                 "so3.condition": False, "conditioning.conditioning_modality": "", "valid.inv_kin": False})
    np.random.seed(SEED), torch.manual_seed(SEED)
    trainer = getattr(importlib.import_module("hp.core.trainer"), cfg.trainer)(cfg)  # only_eval: loads no data
    memorized = torch.utils.data.Subset(memorize_pool, np.arange(n))
    trainer.only_eval = False
    trainer.dataset = trainer.train_dataset = trainer.val_dataset = memorized
    trainer.create_optimizer_and_scheduler()
    trainer.create_dataloaders()
    assert trainer.max_iter == MAX_ITERATIONS, f"the scheduler caps the training at {trainer.max_iter} iterations"

    history = []
    memorized_data = AMASS(data=pool[order[:n]])

    def validate():
        ll_mem, nan_mem = mean_log_likelihood(trainer, memorized_data)
        ll_held, nan_held = mean_log_likelihood(trainer, held_out)
        history.append(dict(iteration=(len(history) + 1) * VALID_INTERVAL, ll_memorized=ll_mem, ll_held_out=ll_held,
                            nan=nan_mem + nan_held))
        return {"nll": -ll_mem, "d_wasser": -1}

    trainer.validate = validate
    start = time.time()
    trainer.train()
    minutes = (time.time() - start) / 60
    last = history[-1]
    print(f"n = {n:6,}: memorized {last['ll_memorized']:9.2f} | held out {last['ll_held_out']:9.2f} | "
          f"NaN {sum(r['nan'] for r in history)} | {minutes:.1f} min")
    return dict(n=n, batch_size=batch, history=history, minutes=minutes)


results = {}
for n in SET_SIZES:
    results[n] = run(n)

## Result

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for k, (n, r) in enumerate(results.items()):
    it = [h["iteration"] for h in r["history"]]
    ax.plot(it, [h["ll_memorized"] for h in r["history"]], color=f"C{k}", label=f"n = {n:,}: memorized")
    ax.plot(it, [h["ll_held_out"] for h in r["history"]], color=f"C{k}", ls="--", label=f"n = {n:,}: held out")
ax.set_xlabel("iteration")
ax.set_ylabel("log-likelihood per pose")# (higher is better)")
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "memorization_test.png", dpi=120)
plt.show()

print(f"after {MAX_ITERATIONS} iterations:")
print(f"{'n':>8s} {'memorized':>10s} {'held out':>10s} {'gap':>8s} {'min':>6s}")
for n, r in results.items():
    last = r["history"][-1]
    print(f"{n:8,} {last['ll_memorized']:10.2f} {last['ll_held_out']:10.2f} {last['ll_memorized'] - last['ll_held_out']:8.2f} {r['minutes']:6.1f}")
if any(h["nan"] for r in results.values() for h in r["history"]):
    print("WARNING: some log-likelihoods were NaN (the training loss ignores them).")

with open(OUT_DIR / "memorization_test.json", "w") as f:
    json.dump(dict(params=dict(set_sizes=SET_SIZES, n_held_out=N_HELD_OUT, batch_size=BATCH_SIZE, max_iterations=MAX_ITERATIONS,
                               valid_interval=VALID_INTERVAL, seed=SEED, data_seed=DATA_SEED, rotations=str(ROTATIONS)),
                   results={str(n): r for n, r in results.items()}), f, indent=1)
print("saved to", OUT_DIR)